# Ground rules

Using Server "Galaktische Republik" for testing

Step 1: Data Preparation


In [168]:
import os.path
import pandas as pd

test_data_path = os.path.join('data', 'dev')
dev_config_path = os.path.join(test_data_path, 'config.json')
dev_event_log_path = os.path.join(test_data_path, 'event_log.csv')


In [169]:
df_voice_events = pd.read_csv(dev_event_log_path)

In [172]:
print(df_voice_events)

   member_id   member_name  timestamp  guild_id guild_name  channel_id  \
0          1      testuser          1         1     server           1   
1          1      testuser          5         1     server           1   
2          1      testuser          5         1     server           2   
3          1      testuser         10         1     server           2   
4          1      testuser         15         1     server           1   
5          2  testuserzwei          5         1     server           1   
6          2  testuserzwei         10         1     server           1   
7          2  testuserzwei         15         1     server           1   
8          2  testuserzwei         20         1     server           1   
9          2  testuserzwei         25         1     server           1   

      channel_name event_type  
0      testchannel       join  
1      testchannel      leave  
2  testchannelzwei       join  
3  testchannelzwei      leave  
4      testchannel       

## Data Preparation

We need to do some fun things

- Convert timestamps into pandas datetimes
- Set member names to every members last known name
- Set channel names to every channels last known name
- Handle orphan joins
- Save Server name once and drop it from the table

### Prepare a copy of the dataframe

In [173]:
df_voice_events_prepped = df_voice_events.copy()

### Convert the timestamps into pandas datetimes

In [174]:
df_voice_events_prepped['timestamp'] = pd.to_datetime(df_voice_events_prepped['timestamp'], unit='s')

### Save the server name plus id and then drop them from the table

In [175]:
last_row = df_voice_events_prepped.sort_values(by=['timestamp'], ascending=True).tail(1)
guild_id = last_row['guild_id'].values[0]
guild_name = last_row['guild_name'].values[0]
print(guild_id)
print(guild_name)
df_voice_events_prepped.drop(columns=['guild_id', 'guild_name'], inplace=True)

1
server


### Set the members names to their last known name, identified by id

In [176]:
member_ids = df_voice_events_prepped['member_id'].unique()
print(f'There are {len(member_ids)} members in this guild')
print(f'Members by id: {df_voice_events_prepped['member_id'].nunique()}, by name: {df_voice_events_prepped['member_name'].nunique()}')
member_id_name_map = {}

for m_id in member_ids:
    df_member = df_voice_events_prepped[df_voice_events_prepped['member_id'] == m_id]
    last_row = df_member.sort_values(by=['timestamp'], ascending=True).tail(1)
    last_known_name = last_row['member_name'].values[0]
    member_id_name_map[m_id] = last_known_name

df_voice_events_prepped.drop(columns=['member_name'], inplace=True)
print(member_id_name_map)

There are 2 members in this guild
Members by id: 2, by name: 2
{np.int64(1): 'testuser', np.int64(2): 'testuserzwei'}


### Do the same for the channels

In [177]:
channel_ids = df_voice_events_prepped['channel_id'].unique()
print(f'There are {len(channel_ids)} used channels in this guild')
print(f'Channels by id: {df_voice_events_prepped['channel_id'].nunique()}, by name: {df_voice_events_prepped['channel_name'].nunique()}')
channel_id_name_map = {}
for c_id in channel_ids:
    df_channel = df_voice_events_prepped[df_voice_events_prepped['channel_id'] == c_id]
    last_row = df_channel.sort_values(by=['timestamp'], ascending=True).tail(1)
    last_known_name = last_row['channel_name'].values[0]
    channel_id_name_map[c_id] = last_known_name

df_voice_events_prepped.drop(columns=['channel_name'], inplace=True)

print(channel_id_name_map)

There are 2 used channels in this guild
Channels by id: 2, by name: 2
{np.int64(1): 'testchannel', np.int64(2): 'testchannelzwei'}


### Before Orphan fixing let us take a snapshot to see if all goes well

In [178]:
print(df_voice_events_prepped)

def verify_event_counts(df):
    counts = df.groupby(['member_id', 'event_type']).size().unstack(fill_value=0)
    print(counts)
    return (counts['join'] == counts['leave']).all()

print(verify_event_counts(df_voice_events_prepped))


   member_id           timestamp  channel_id event_type
0          1 1970-01-01 00:00:01           1       join
1          1 1970-01-01 00:00:05           1      leave
2          1 1970-01-01 00:00:05           2       join
3          1 1970-01-01 00:00:10           2      leave
4          1 1970-01-01 00:00:15           1       join
5          2 1970-01-01 00:00:05           1      leave
6          2 1970-01-01 00:00:10           1       join
7          2 1970-01-01 00:00:15           1      leave
8          2 1970-01-01 00:00:20           1       join
9          2 1970-01-01 00:00:25           1      leave
event_type  join  leave
member_id              
1              3      2
2              2      3
False


### Handle orphan joins

Inserted 6 missing event(s):


,member_id,channel_id,timestamp,event_type,reason
0,124567572397031425,1370422030939328675,2026-02-08 23:13:18.145958185,join,missing 'join' before 'leave'
1,286156392312733697,1370422030939328675,2026-02-08 21:24:16.001079798,join,missing 'join' before 'leave'
2,474281490822463488,963198266600722532,2026-08-07 22:49:58.627666712,leave,trailing 'join' with no closing 'leave' at end...
3,691704924693594125,963198266600722532,2026-08-07 22:19:11.810528517,leave,trailing 'join' with no closing 'leave' at end...
4,770367618921398290,1370422030939328675,2026-02-09 23:37:52.313694954,leave,missing 'leave' before 'join'
5,796837302096887879,963198266600722532,2026-08-07 22:55:57.541040897,leave,trailing 'join' with no closing 'leave' at end...


In [156]:

print(verify_event_counts(df_voice_events_prepped))


event_type           join  leave
member_id                       
124567572397031425    183    184
176034509014171648    274    274
185812748192448512     64     64
286156392312733697     13     14
372767039657345036     68     68
442768279388291072    195    195
474281490822463488     78     77
608068617069789369    195    195
675740751199600671     65     65
691704924693594125     54     53
713465732704501790     56     56
770367618921398290    137    136
796837302096887879     32     31
1007986223631048704    38     38
1253824766507225179     1      1
1319315891355258990     4      4
1505237993948971008    46     46
1523070486236365007     7      7
False
